# 🎬 FILM AGENT — Character Consistency LoRA Fine-Tuning Notebook
### (`arjun_male` & `maya_female` 360° Turnaround Multi-Angle LoRA Trainer)

මෙම Notebook එක මඟින් ඔබගේ `d:\film-argent\character_lora_dataset.zip` (Turnaround Sheets දෙකෙන් කපාගත් Angles 26 + Captions) පාවිච්චි කර **Character Consistency LoRA (`film_characters_lora.safetensors`)** ෆයිල් එක Train කර ලබාදෙයි.

### 📌 Steps:
1. Google Colab වෙත මෙම `.ipynb` ෆයිල් එක Upload කරන්න.
2. ඉහළ මෙනුවෙන් **Runtime ➔ Change runtime type ➔ T4 GPU** තෝරන්න.
3. **Cell 1** සිට **Cell 5** දක්වා පිළිවෙළින් Run කරන්න.

In [ ]:
# @title 1️⃣ Cell 1: GPU පරීක්ෂාව සහ Libraries Install කිරීම
!nvidia-smi
!pip install -q -U diffusers transformers accelerate peft bitsandbytes safetensors datasets "pillow<13.0" "pyarrow<24.0" tqdm
print('✅ සියලුම Libraries සාර්ථකව Install විය!')

In [ ]:
# @title 2️⃣ Cell 2: `character_lora_dataset.zip` ෆයිල් එක Upload කිරීම
# ඔබගේ PC එකේ d:\film-argent\character_lora_dataset.zip ෆයිල් එක මෙතැනින් Upload කරන්න
import os, zipfile
from google.colab import files

DATASET_DIR = '/content/lora_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

print('📂 කරුණාකර d:\\film-argent\\character_lora_dataset.zip ෆයිල් එක තෝරන්න...')
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(DATASET_DIR)
        print(f'✅ Extracted {fname} to {DATASET_DIR}')

# Count extracted images and captions
imgs = [os.path.join(r, f) for r, _, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(('.png', '.jpg', '.jpeg'))]
print(f'🎯 Total Training Images Ready: {len(imgs)} (arjun_male + maya_female multi-angle views)')

In [ ]:
# @title 3️⃣ Cell 3: HuggingFace Diffusers LoRA Training Script සූදානම් කිරීම
!git clone -q https://github.com/huggingface/diffusers.git /content/diffusers_repo
!pip install -q -e /content/diffusers_repo

import os, json, shutil
from pathlib import Path

FLAT_DATA_DIR = Path('/content/flat_training_data')
if FLAT_DATA_DIR.exists():
    shutil.rmtree(FLAT_DATA_DIR)
FLAT_DATA_DIR.mkdir(parents=True, exist_ok=True)

metadata_lines = []
for root, _, files_list in os.walk('/content/lora_dataset'):
    for f in sorted(files_list):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_src = Path(root) / f
            txt_src = img_src.with_suffix('.txt')
            caption = txt_src.read_text(encoding='utf-8').strip() if txt_src.exists() else 'photo of arjun_male and maya_female'
            # Repeat 4x to give strong multi-angle convergence
            for rep in range(4):
                dest_name = f'rep{rep}_{f}'
                shutil.copy(img_src, FLAT_DATA_DIR / dest_name)
                metadata_lines.append({'file_name': dest_name, 'prompt': caption})

with open(FLAT_DATA_DIR / 'metadata.jsonl', 'w', encoding='utf-8') as mf:
    for item in metadata_lines:
        mf.write(json.dumps(item) + '\n')

print(f'✅ Dataset formatted with {len(metadata_lines)} training samples (Trigger words: arjun_male, maya_female)!')

In [ ]:
# @title 4️⃣ Cell 4: Character LoRA Fine-Tuning ආරම්භ කිරීම (~15-20 mins on T4 GPU)
import os
OUTPUT_DIR = '/content/output_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

!accelerate launch /content/diffusers_repo/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/content/flat_training_data" \
  --caption_column="prompt" \
  --resolution=1024 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=25 \
  --mixed_precision="fp16" \
  --seed=42 \
  --output_dir="/content/output_lora" \
  --rank=32 \
  --gradient_checkpointing \
  --use_8bit_adam

print('🎉 LoRA Fine-Tuning Completed!')

In [ ]:
# @title 5️⃣ Cell 5: Train වුණු `film_characters_lora.safetensors` ෆයිල් එක Download කරගැනීම
# මෙය Download වූ පසු d:\film-argent\backend\models\film_characters_lora.safetensors ලෙස දමන්න!
import shutil
from pathlib import Path
from google.colab import files

src_weights = Path('/content/output_lora/pytorch_lora_weights.safetensors')
final_lora = Path('/content/film_characters_lora.safetensors')

if src_weights.exists():
    shutil.copy(src_weights, final_lora)
    print(f'✅ Ready: {final_lora} ({round(final_lora.stat().st_size / (1024*1024), 2)} MB)')
    files.download(str(final_lora))
else:
    print('❌ LoRA weights file not found. Check Cell 4 output logs.')